In [ ]:
# Frenny Montezuma Castillo
# Grupo 04
from pandas.core.dtypes.missing import notna
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

uploaded = files.upload()
nombre_archivo = list(uploaded.keys())[0]
df = pd.read_excel(nombre_archivo, sheet_name=0)
df = df.dropna(axis=1, how="all")
df.columns = [str(c).strip() for c in df.columns]
col_fecha = [c for c in df.columns if "Fecha" in c][0]
df[col_fecha] = pd.to_datetime(df[col_fecha])
df = df.sort_values(col_fecha).reset_index(drop=True)
cols_numeros = [c for c in df.columns if "Número" in c and "Sorteo" not in c]
print(f"{len(df)} sorteos cargados, del {df[col_fecha].min().date()} al {df[col_fecha].max().date()}")

rango_numeros = list(range(100))
binaria = pd.DataFrame(0, index=df.index, columns=rango_numeros)
for c in cols_numeros:
    for idx, val in df[c].items():
        if pd.notna(val):
            binaria.loc[idx, int(val)] = 1

VENTANAS = [5, 15, 30]
N_SORTEOS = 500
muestra = binaria.tail(N_SORTEOS).reset_index(drop=True)

dict_features = {}
for num in rango_numeros:
    s = muestra[num]
    for v in VENTANAS:
        dict_features[f"freq_{num}_v{v}"] = s.rolling(v, min_periods=1).mean().shift(1).fillna(0)

X = pd.DataFrame(dict_features)
y = muestra

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

modelo = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1),
    n_jobs=-1
)
modelo.fit(X_train, y_train)

probs_list = modelo.predict_proba(X_test)
probs_mat = np.column_stack([p[:, 1] if p.shape[1] > 1 else np.zeros(len(X_test)) for p in probs_list])

TOP_K = 5
aciertos_modelo = []
num_ganadores_por_sorteo = y_test.sum(axis=1).values

for i in range(len(y_test)):
    top_k_pred = np.argsort(probs_mat[i])[::-1][:TOP_K]
    reales = np.where(y_test.iloc[i].values == 1)[0]
    aciertos = len(set(top_k_pred).intersection(set(reales)))
    aciertos_modelo.append(aciertos)

promedio_aciertos = np.mean(aciertos_modelo)
azar_esperado = (np.mean(num_ganadores_por_sorteo) * TOP_K) / 100

print(f"--- RESULTADOS DE EVALUACIÓN ---")
print(f"Promedio de aciertos del modelo (Top {TOP_K}): {promedio_aciertos:.2f} números por sorteo")
print(f"Aciertos esperados por puro azar (Top {TOP_K}): {azar_esperado:.2f} números por sorteo")
print(f"Tasa de efectividad relativa: {(promedio_aciertos / max(azar_esperado, 1e-5)):.2f}x sobre el azar")

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(14, 6))

x_axis = np.arange(len(aciertos_modelo))
scatter = ax.scatter(x_axis, aciertos_modelo, c=aciertos_modelo, cmap='plasma', s=60, zorder=3, edgecolors='none')

ax.plot(x_axis, aciertos_modelo, color='#00ffcc', alpha=0.4, linewidth=1.5, zorder=2)
ax.axhline(y=azar_esperado, color='#ff0055', linestyle='--', linewidth=2, label=f'Azar Esperado ({azar_esperado:.2f})', zorder=4)
ax.axhline(y=promedio_aciertos, color='#00e5ff', linestyle=':', linewidth=2, label=f'Promedio Modelo ({promedio_aciertos:.2f})', zorder=4)

ax.set_title(f'Rendimiento del Modelo en Tiempo Real (Top {TOP_K} Predicciones)', fontsize=14, color='white', pad=15)
ax.set_xlabel('Sorteo Evaluación (Conjunto de Test)', fontsize=11, color='white')
ax.set_ylabel('Aciertos por Sorteo', fontsize=11, color='white')
ax.grid(True, color='#333333', linestyle='--', alpha=0.6)
ax.legend(facecolor='#111111', edgecolor='#444444', loc='upper right')

cbar = plt.colorbar(scatter)
cbar.set_label('Escala de Aciertos', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

plt.tight_layout()
plt.show()